# 06 GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted. Notebook 06 builds SI from all screened yearly pairs and applies its own explicit fit and plot filters, independently of notebook 05 pair-selection thresholds.

## Inputs

- Screened yearly pairs and background publication counts from notebook 04.
- GLASS pair data and the configured pathogen/antibiotic lookup tables.

## Outputs

- Notebook 06 SI/rank caches, interactive GLASS comparison figures, PDF exports, and supporting CSV tables under the configured `outputs/` directory.


In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    build_yearly_si_dataset,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    load_yearly_si_inputs,
    prepare_antibiotic_analysis_level,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Parameters

In [35]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
FIGURE_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.figure_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)

ANTIBIOTIC_ANALYSIS_LEVEL = cfg(PIPELINE_CONFIG, "glass_interactive.antibiotic_analysis_level")
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
PATHOGEN_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_background", PROJECT_ROOT)
ANTIBIOTIC_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_background", PROJECT_ROOT)
YEARLY_SI_NB06_PATH = SI_PATHS["yearly_si"].with_name("yearly_si_nb06.parquet")
# Notebook 06 has independent caches; it does not overwrite notebook 05 outputs.
RANK_OUTPUT_PATH = SI_PATHS["rank_output"].with_name("yearly_si_ranked_future_nb06.parquet")

GLASS_XLSX_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_xlsx", PROJECT_ROOT)
GLASS_CLEANED_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_cleaned", PROJECT_ROOT)
BACTERIA_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)

# Model/forecast settings. These define the fit window and curve, not Notebook 05's
# interesting-pair selection.
FIT_START_YEAR = cfg(PIPELINE_CONFIG, "si.fit_start_year")
FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.fit_end_year")
CURVE_FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.curve_fit_end_year")
NUM_FUTURE_YEARS = cfg(PIPELINE_CONFIG, "glass_interactive.num_future_years")
LOESS_FRAC = cfg(PIPELINE_CONFIG, "si.loess_frac")
LOESS_DEGREE = cfg(PIPELINE_CONFIG, "si.loess_degree")
TAIL_DAMPING_YEARS = cfg(PIPELINE_CONFIG, "si.tail_damping_years")

# Notebook 06 pair-fit eligibility. These are local controls, independent of
# notebook 05's thresholds for selecting interesting pairs.
MIN_AMR_PAIR_PUBLICATION_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_amr_pair_publication_count")
MIN_NUM_YEARS_WITH_VALID_COUNTS = cfg(PIPELINE_CONFIG, "glass_interactive.min_num_years_with_valid_counts")

# Notebook 06 plot inclusion criteria, applied after fitting.
MIN_PLOT_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_plot_si")
MIN_LAST_AVAILABLE_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_last_available_count")
MIN_TOTAL_PUBLICATIONS = cfg(PIPELINE_CONFIG, "glass_interactive.min_total_publications")
MIN_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.min_count_abs_pct_error_mean")
MAX_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.max_count_abs_pct_error_mean")
MIN_SI_DELTA = cfg(PIPELINE_CONFIG, "glass_interactive.min_si_delta")

# Plot presentation controls (not pair eligibility filters).
PREDICTED_FUTURE_SI_THRESHOLD = cfg(PIPELINE_CONFIG, "glass_interactive.predicted_future_si_threshold")
PREDICTED_FUTURE_DELTA_SI_THRESHOLD = cfg(PIPELINE_CONFIG, "glass_interactive.predicted_future_delta_si_threshold")
SHOW_MEAN_COUNT_PCT_ERROR = cfg(PIPELINE_CONFIG, "glass_interactive.show_mean_count_pct_error")
SHOW_LABEL_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.show_label_background")
MIN_COUNTS_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.min_counts_background")
MIN_ABS_DELTA_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_abs_delta_si")
DISPLAY_NAME = cfg(PIPELINE_CONFIG, "glass_interactive.display_name")
MAX_TEXT_LABELS = cfg(PIPELINE_CONFIG, "glass_interactive.max_text_labels")
WIDTH = cfg(PIPELINE_CONFIG, "glass_interactive.width")
HEIGHT = cfg(PIPELINE_CONFIG, "glass_interactive.height")

for required_input_path in (YEARLY_SCREENED_PATH, PATHOGEN_BG_PATH, ANTIBIOTIC_BG_PATH):
    if not required_input_path.exists():
        raise FileNotFoundError(
            f"Missing {required_input_path}. Run notebook 04 to create screened AMR and background counts first."
        )


## Load SI And Clean GLASS List

In [36]:
yearly_screened_df, pathogen_bg_df, antibiotic_bg_df, _ = load_yearly_si_inputs(
    yearly_screened_path=YEARLY_SCREENED_PATH,
    pathogen_background_path=PATHOGEN_BG_PATH,
    antibiotic_background_path=ANTIBIOTIC_BG_PATH,
)

yearly_screened_model_df, antibiotic_bg_model_df, _, analysis_summary = prepare_antibiotic_analysis_level(
    yearly_screened_df=yearly_screened_df,
    antibiotic_background_df=antibiotic_bg_df,
    unified_decisions_df=pd.DataFrame(),
    coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    analysis_level=ANTIBIOTIC_ANALYSIS_LEVEL,
)
si_df, _, _ = build_yearly_si_dataset(
    yearly_screened_model_df,
    pathogen_bg_df,
    antibiotic_bg_model_df,
    min_amr_pair_publication_count=MIN_AMR_PAIR_PUBLICATION_COUNT,
    output_path=YEARLY_SI_NB06_PATH,
)
_, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_NB06_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=yearly_screened_model_df,
    modeled_df=si_df,
    source_label="all screened AMR input pairs",
    modeled_label="Notebook 06 yearly SI input",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

yearly_screened rows: 64,721
yearly_screened unique pathogens: 719
yearly_screened unique antibiotics: 213
yearly_screened year range: 2000 - 2026
pathogen bg rows: 21,330
antibiotic bg rows: 5,886
unified decision rows available for driver analysis: 0
Antibiotic analysis level: individual antibiotic.
model yearly rows: 64,721
model pathogen-antibiotic rows: 15,539
model antibiotic values: 213
model antibiotic background rows: 5,886
model unified decision rows: 0
Loaded SI rows: 64,721 from /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si_nb06.parquet
Pathogens: 719; antibiotics/levels: 213
Saved cleaned GLASS rows: 224 to /Users/f.costa/Code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
all screened AMR input pairs pathogens: 719
Notebook 06 yearly SI input pathogens: 719


,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,Acinetobacter spp.,amikacin,Acinetobacter bereziniae,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
9,Acinetobacter spp.,Colistin,Acinetobacter bereziniae,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified


## Build Or Load Rank Data

In [37]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_NB06_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    min_amr_pair_publication_count=MIN_AMR_PAIR_PUBLICATION_COUNT,
    min_num_years_with_valid_counts=MIN_NUM_YEARS_WITH_VALID_COUNTS,
    verbose=True,
)

num_ranked_pairs = len(rank_df)
rank_df = rank_df.loc[rank_df["status"].eq("ok")].copy()
num_successful_fits = len(rank_df)
rank_df = rank_df.loc[
    pd.to_numeric(rank_df["latest_available_predicted_SI"], errors="coerce") >= MIN_PLOT_SI
].copy()
print(f"Successfully fitted pairs: {num_successful_fits:,} / {num_ranked_pairs:,}")
print(f"Pairs with latest predicted SI >= {MIN_PLOT_SI:g}: {len(rank_df):,}")
display(rank_df)

Recomputing rank data because /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future_nb06.parquet is older than /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si_nb06.parquet
[1/15539] ranking Abiotrophia defectiva vs Amoxicillin
[2/15539] ranking Abiotrophia defectiva vs Ampicillin
[3/15539] ranking Abiotrophia defectiva vs Benzylpenicillin
[4/15539] ranking Abiotrophia defectiva vs Cefaclor
[5/15539] ranking Abiotrophia defectiva vs Cefepime
[6/15539] ranking Abiotrophia defectiva vs Cefotaxime
[7/15539] ranking Abiotrophia defectiva vs Ceftazidime
[8/15539] ranking Abiotrophia defectiva vs Ceftriaxone
[9/15539] ranking Abiotrophia defectiva vs Ciprofloxacin
[10/15539] ranking Abiotrophia defectiva vs Clindamycin
[11/15539] ranking Abiotrophia defectiva vs Dalfopristin/quinupristin
[12/15539] ranking Abiotrophia defectiva vs Daptomycin
[13/15539] ranking Abiotrophia defectiva vs Erythromycin
[14/15539] ranking Abiotrophia defectiva

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,min_amr_pair_publication_count,...,tail_damping_years,validation_loess_frac,validation_loess_degree,validation_tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
13492,Mycobacterium kansasii,Clarithromycin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,734.0,1463.0,-729.0,729.0
13493,Mycobacterium tuberculosis,Ofloxacin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,562.0,1263.0,-701.0,701.0
13494,Staphylococcus capitis,Linezolid,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,1115.0,1813.0,-698.0,698.0
13495,Klebsiella pneumoniae,Neomycin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,1617.0,926.0,691.0,691.0
13496,Corynebacterium striatum,Daptomycin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,1080.0,1735.0,-655.0,655.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15534,Streptococcus pneumoniae,Tebipenem,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,2042.0,2042.0,0.0,0.0
15535,Bacillus subtilis,Fosfomycin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,2044.0,2044.0,0.0,0.0
15536,Mycobacterium tuberculosis,Trimethoprim,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,2045.0,2045.0,0.0,0.0
15537,Haemophilus influenzae,Faropenem,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,2046.0,2046.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [38]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_total_publications=MIN_TOTAL_PUBLICATIONS,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

Displayed rows: 324; GLASS matches: 55
Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/interactive_latest_observed_count_vs_future_si_shift.pdf


## Display GLASS Matches In Plot

In [39]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Escherichia coli,Ciprofloxacin,2025.0,2028,266.0,3589.0,292.564235,1.165247,1.222699,0.057452,9.713938,antibiotic
1,Klebsiella pneumoniae,Ceftazidime,2025.0,2028,188.0,1754.0,353.731821,1.181655,1.365974,0.184319,15.726750,antibiotic
2,Klebsiella pneumoniae,Meropenem,2025.0,2028,185.0,1957.0,286.402388,1.200874,1.301847,0.100973,20.622386,antibiotic
3,Klebsiella pneumoniae,Colistin,2025.0,2028,176.0,1958.0,267.891130,1.172041,1.248582,0.076541,42.629825,antibiotic
4,Pseudomonas aeruginosa,Ceftazidime,2025.0,2028,159.0,2141.0,200.147834,1.094827,1.167572,0.072746,14.838768,antibiotic
5,Escherichia coli,Meropenem,2025.0,2028,158.0,1635.0,265.522775,1.068748,1.205308,0.136560,25.445255,antibiotic
6,Pseudomonas aeruginosa,Meropenem,2025.0,2028,154.0,1955.0,202.856863,1.121580,1.200665,0.079086,5.133357,antibiotic
7,Acinetobacter baumannii,Colistin,2025.0,2028,151.0,1995.0,168.634609,1.151924,1.172781,0.020857,5.289815,antibiotic
8,Pseudomonas aeruginosa,Colistin,2025.0,2028,138.0,1477.0,165.429182,1.013686,1.088227,0.074540,2.591584,antibiotic
9,Escherichia coli,Sulfamethoxazole/trimethoprim,2025.0,2028,125.0,1882.0,168.913585,0.997864,1.060396,0.062531,10.599733,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [40]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 269


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Klebsiella pneumoniae,Amoxicillin/clavulanic-acid,2025.0,2028,35.0,208.0,54.250044,0.512163,0.734561,0.222398,20.977684
1,Escherichia coli,Vancomycin,2025.0,2028,16.0,116.0,21.919774,-0.073307,0.130679,0.203986,35.132459
2,Klebsiella pneumoniae,Nitrofurantoin,2025.0,2028,29.0,210.0,48.978607,0.521511,0.717991,0.196480,11.757613
3,Escherichia coli,Ceftazidime/avibactam,2025.0,2028,45.0,288.0,72.326473,0.443467,0.638426,0.194959,15.781442
4,Escherichia coli,Cefixime,2025.0,2028,24.0,184.0,37.631235,0.441879,0.626627,0.184748,4.596345
...,...,...,...,...,...,...,...,...,...,...,...
264,Enterococcus faecium,Gentamicin,2025.0,2028,7.0,314.0,8.403372,0.204864,0.126387,-0.078477,60.308421
265,Klebsiella pneumoniae,Ceftolozane/tazobactam,2025.0,2028,9.0,118.0,8.920271,0.128149,0.048426,-0.079723,38.584789
266,Staphylococcus aureus,Nitrofurantoin,2025.0,2028,7.0,101.0,4.059952,-0.245832,-0.333465,-0.087634,29.508184
267,Mycobacterium tuberculosis,Kanamycin,2025.0,2028,12.0,364.0,9.761288,0.388959,0.291892,-0.097067,16.901991


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [41]:
latest_count_shift_df

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,min_amr_pair_publication_count,...,latest_count_source,latest_smoothed_SI,future_minus_latest_smoothed_SI,count_error_display,marker_count_error,is_glass_pair,glass_group_column,appears_in_GLASS,label,display_label
0,Escherichia coli,Vancomycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.073307,0.203986,35.13%,35.132459,False,antibiotic,no,E. coli<br>Vancomycin,E. coli<br>Vancomycin
1,Staphylococcus aureus,Dalbavancin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.027139,-0.107018,46.41%,46.413834,False,antibiotic,no,S. aureus<br>Dalbavancin,S. aureus<br>Dalbavancin
2,Klebsiella pneumoniae,Ceftolozane/tazobactam,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.128149,-0.079723,38.58%,38.584789,False,antibiotic,no,K. pneumoniae<br>Ceftolozane/tazobactam,K. pneumoniae<br>Ceftolozane/tazobactam
3,Enterococcus faecium,Gentamicin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.204864,-0.078477,60.31%,60.308421,False,antibiotic,no,E. faecium<br>Gentamicin,E. faecium<br>Gentamicin
4,Staphylococcus aureus,Ceftazidime,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.143466,0.048217,65.33%,65.330130,False,antibiotic,no,S. aureus<br>Ceftazidime,S. aureus<br>Ceftazidime
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,Klebsiella pneumoniae,Meropenem,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,1.200874,0.100973,20.62%,20.622386,True,antibiotic,yes,K. pneumoniae<br>Meropenem,K. pneumoniae<br>Meropenem
320,Klebsiella pneumoniae,Colistin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,1.172041,0.076541,42.63%,42.629825,True,antibiotic,yes,K. pneumoniae<br>Colistin,K. pneumoniae<br>Colistin
321,Escherichia coli,Sulfamethoxazole/trimethoprim,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.997864,0.062531,10.60%,10.599733,True,antibiotic,yes,E. coli<br>Sulfamethoxazole/trimethoprim,E. coli<br>Sulfamethoxazole/trimethoprim
322,Pseudomonas aeruginosa,Tobramycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.859170,0.054290,16.17%,16.169963,True,antibiotic,yes,P. aeruginosa<br>Tobramycin,P. aeruginosa<br>Tobramycin


In [46]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    predicted_future_si_threshold=0.0,
    predicted_future_delta_si_threshold=0.0,
    show_mean_count_pct_error=SHOW_MEAN_COUNT_PCT_ERROR,
    show_label_background=SHOW_LABEL_BACKGROUND,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    avoid_label_occlusion=True,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_current_si_quadrant.pdf",
    quadrant_output_dir=FIGURE_OUTPUT_DIR / "future_si_quadrants",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")
for quadrant_name, quadrant_result in future_si_shift_result["quadrants"].items():
    if quadrant_result["pdf_path"]:
        print(f"Saved {quadrant_name} quadrant: {quadrant_result['pdf_path']}")
    elif quadrant_result.get("pdf_error"):
        print(f"PDF export failed for {quadrant_name}: {quadrant_result['pdf_error']}")

Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_vs_current_si_quadrant.pdf
Saved below_both quadrant: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_quadrants/future_si_vs_si_shift_below_both.pdf
Saved below_future_si quadrant: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_quadrants/future_si_vs_si_shift_below_future_si.pdf
Saved below_delta_si quadrant: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_quadrants/future_si_vs_si_shift_below_delta_si.pdf
Saved above_both quadrant: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_quadrants/future_si_vs_si_shift_above_both.pdf


In [43]:
latest_count_shift_df.to_csv(
    FIGURE_OUTPUT_DIR / "future_si_vs_current_si_quadrant.csv",
    index=False,
)

In [44]:
FIGURE_OUTPUT_DIR / "future_si_vs_current_si_quadrant.csv"

PosixPath('/Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_vs_current_si_quadrant.csv')

In [45]:
latest_count_shift_df[latest_count_shift_df['pathogen'] == 'Streptococcus pneumoniae']

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,min_amr_pair_publication_count,...,latest_count_source,latest_smoothed_SI,future_minus_latest_smoothed_SI,count_error_display,marker_count_error,is_glass_pair,glass_group_column,appears_in_GLASS,label,display_label
10,Streptococcus pneumoniae,Levofloxacin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.251894,-0.055184,40.51%,40.509285,False,antibiotic,no,S. pneumoniae<br>Levofloxacin,S. pneumoniae<br>Levofloxacin
12,Streptococcus pneumoniae,Meropenem,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.157570,0.018551,29.23%,29.229211,False,antibiotic,no,S. pneumoniae<br>Meropenem,S. pneumoniae<br>Meropenem
15,Streptococcus pneumoniae,Vancomycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.146202,-0.019027,35.68%,35.678958,False,antibiotic,no,S. pneumoniae<br>Vancomycin,S. pneumoniae<br>Vancomycin
17,Streptococcus pneumoniae,Azithromycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.102271,-0.028564,14.98%,14.976223,False,antibiotic,no,S. pneumoniae<br>Azithromycin,S. pneumoniae<br>Azithromycin
39,Streptococcus pneumoniae,Sulfamethoxazole/trimethoprim,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.245946,0.003317,51.17%,51.170391,True,antibiotic,yes,S. pneumoniae<br>Sulfamethoxazole/trimethoprim,S. pneumoniae<br>Sulfamethoxazole/trimethoprim
43,Streptococcus pneumoniae,Sulfamethoxazole,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.268437,0.010707,37.65%,37.647684,False,antibiotic,no,S. pneumoniae<br>Sulfamethoxazole,S. pneumoniae<br>Sulfamethoxazole
44,Streptococcus pneumoniae,Trimethoprim,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.249827,0.009149,42.25%,42.250638,False,antibiotic,no,S. pneumoniae<br>Trimethoprim,S. pneumoniae<br>Trimethoprim
46,Streptococcus pneumoniae,Clindamycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.251402,0.010044,47.78%,47.784378,False,antibiotic,no,S. pneumoniae<br>Clindamycin,S. pneumoniae<br>Clindamycin
68,Streptococcus pneumoniae,Erythromycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.592908,-0.019422,14.21%,14.208547,True,antibiotic,yes,S. pneumoniae<br>Erythromycin,S. pneumoniae<br>Erythromycin
78,Streptococcus pneumoniae,Ceftriaxone,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.402023,0.003258,40.47%,40.471286,True,antibiotic,yes,S. pneumoniae<br>Ceftriaxone,S. pneumoniae<br>Ceftriaxone


## 2025 PMIDs For Plotted Pairs

One row per PMID linked to each plotted bacteria-antibiotic pair in 2025. Pairs without a 2025 PMID remain in the table with a blank PMID.

In [48]:
CURRENT_YEAR = 2025

plotted_pairs_df = latest_count_shift_df[["pathogen", "antibiotic"]].drop_duplicates()
pmids_current_year_df = yearly_screened_model_df.loc[
    pd.to_numeric(yearly_screened_model_df["year"], errors="coerce").eq(CURRENT_YEAR),
    ["pathogen", "antibiotic", "pmids"],
].copy()

pair_pmids_2025_df = plotted_pairs_df.merge(
    pmids_current_year_df,
    on=["pathogen", "antibiotic"],
    how="left",
).explode("pmids", ignore_index=True)
pair_pmids_2025_df = (
    pair_pmids_2025_df.rename(columns={"pathogen": "bacteria", "pmids": "PMID"})
    .assign(year=CURRENT_YEAR)
    [["bacteria", "antibiotic", "year", "PMID"]]
    .sort_values(["bacteria", "antibiotic", "PMID"], na_position="last")
    .reset_index(drop=True)
)

from IPython.display import HTML

def _pmid_link(pmid):
    if pd.isna(pmid):
        return ""
    pmid = str(pmid)
    return f'<a href="https://pubmed.ncbi.nlm.nih.gov/{pmid}/" target="_blank">{pmid}</a>'

display(HTML(pair_pmids_2025_df.to_html(index=False, escape=False, formatters={"PMID": _pmid_link})))

bacteria,antibiotic,year,PMID
Acinetobacter baumannii,Amikacin,2025,39471310
Acinetobacter baumannii,Amikacin,2025,39618541
Acinetobacter baumannii,Amikacin,2025,39675013
Acinetobacter baumannii,Amikacin,2025,39704504
Acinetobacter baumannii,Amikacin,2025,39781207
Acinetobacter baumannii,Amikacin,2025,40125744
Acinetobacter baumannii,Amikacin,2025,40150047
Acinetobacter baumannii,Amikacin,2025,40160221
Acinetobacter baumannii,Amikacin,2025,40221655
Acinetobacter baumannii,Amikacin,2025,40330064
